# COMTAIL - Difficulty Assignment (21 Indic translation pairs)

Assigns `difficulty` (Easy / Medium / Hard) to 200 sampled COMTAIL rows.

**Task.** `question` is an English or Hindi sentence, `answer` its translation
into one of 19 Indic languages. The `language` field names the pair
(`eng-tel`, `hin-mar`, ...). Scored with **chrF++**, which the file declares.

**What makes this file different from every other translation split: it holds
21 pairs at once, and they do not behave alike.** Copying the source without
translating scores:

| Pair | Copy-the-source chrF++ |
|---|---|
| `hin-doi` | **33.1%** |
| `hin-mar` | **22.1%** |
| the other 19 pairs | 2 - 6% |

Dogri and Marathi are written in Devanagari and share heavy vocabulary with
Hindi, so a model can score well there by barely changing the input. Every
other pair crosses scripts, where copying earns almost nothing.

A single global floor would therefore be meaningless. **Cell 4 measures the
floor separately for each pair, using every row of that pair in the file** -
not just the sampled ones - and Cell 11 reports accuracy per pair against its
own floor. A model that looks strong overall may simply be doing well on the
two easy-to-copy pairs.

**Two-pass design**, as in the other translation notebooks:

1. Score every row with every model, storing the raw chrF++ value.
2. Derive a threshold (Cell 9); a model passes a row when it clears it, and the
   three votes sum:

| Models passing | Difficulty |
|---|---|
| 3 / 3 | Easy |
| 2 / 3 | Medium |
| 0-1 / 3 | Hard |

**`PAIR_FILTER` in Cell 3** restricts the run to one pair, or to all `eng-*` or
all `hin-*`. With 200 rows spread over 21 pairs you get roughly 10 rows each -
enough to spot a broken pair, not enough to characterise one. Filter when you
want a real read on a specific pair.

**Output.** `comtail_<scope>_difficulty.jsonl` - all 14 schema fields with
`difficulty` filled in, plus an audit file naming each row's pair.

### Cell 1 - Install dependencies and authenticate

`sacrebleu` provides the reference chrF++ implementation.

**An HF token is required.** Llama-3.1 and Gemma-2 are gated; Mistral is not.
Accept both licences on huggingface.co, create a **read** token, then add it in
Colab via the **key icon** as a secret named `HF_TOKEN`.

In [1]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub sacrebleu

import sacrebleu
print("sacrebleu", sacrebleu.__version__)

HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("No HF token ({}: {})".format(type(e).__name__, e))
    print("Mistral will still work; gated Llama/Gemma will fail with a 401.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 10.1 MB/s eta 0:00:00
sacrebleu 2.6.0
HF login OK


### Cell 2 - Mount Drive

Drive is the weight cache: three models, about **16.5 GB**. If you have run any
of the other difficulty notebooks these are already cached and nothing is
downloaded.

In [4]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
else:
    print("\nWARNING: no Drive - weights will NOT be cached between sessions.")

Mounted at /drive
Drive mounted | weight cache: /drive/MyDrive/models


### Cell 3 - Configuration

- `PAIR_FILTER` - `None` for all 21 pairs, `"eng"` or `"hin"` for one source
  language, or an exact pair such as `"hin-mar"`. Output paths derive from it,
  so filtered runs never overwrite the full one.
- `THRESHOLD_MODE` - `"auto_median"` by default. `"floor_margin"` anchors to
  the **highest per-pair floor present in the sample**, which is the safe
  choice when `hin-doi` or `hin-mar` are included.
- `LANG_NAMES` maps the corpus's three-letter codes to language names for the
  prompt - the model is told which languages it is working between.

In [5]:
import gc
import re
import json
import random
import shutil
import statistics
from collections import Counter, defaultdict

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

INPUT_FILE = "comtail.jsonl"

# ---- scope ----
PAIR_FILTER = None          # None | "eng" | "hin" | e.g. "hin-mar"

_scope = PAIR_FILTER if PAIR_FILTER else "all"
OUTPUT_FILE = "comtail_{}_difficulty.jsonl".format(_scope.replace("-", "_"))
AUDIT_FILE  = "comtail_{}_audit.jsonl".format(_scope.replace("-", "_"))
PROG_DIR    = "judge_progress_comtail_{}".format(_scope.replace("-", "_"))

# ---- sampling ----
N_ROWS        = 200
SEED          = 42
MIN_REF_WORDS = 3

# ---- thresholding ----
THRESHOLD_MODE  = "auto_median"     # auto_median | floor_margin | fixed
FLOOR_MARGIN    = 0.15
FIXED_THRESHOLD = 0.50

# ---- generation ----
MAX_NEW_TOKENS = 256
BATCH_SIZE     = 25

# ---- schema ----
SET_EVAL_METRIC = None              # None keeps the file's own "chrF++"

LANG_NAMES = {
    "eng": "English", "hin": "Hindi", "ban": "Bengali", "guj": "Gujarati",
    "kan": "Kannada", "kas": "Kashmiri", "mar": "Marathi", "odi": "Odia",
    "pan": "Punjabi", "tam": "Tamil", "tel": "Telugu", "urd": "Urdu",
    "doi": "Dogri", "snd": "Sindhi",
}

# pairs where source and target share a script AND close vocabulary, so
# copying the input already scores well - measured, not assumed
HIGH_COPY_PAIRS = {"hin-doi", "hin-mar"}

# ---- the three judges ----
USE_INSTRUCT = True
REPOS = {
    True: {"mistral": "mistralai/Mistral-7B-Instruct-v0.3",
           "llama":   "meta-llama/Llama-3.1-8B-Instruct",
           "gemma":   "google/gemma-2-9b-it"},
    False: {"mistral": "mistralai/Mistral-7B-v0.3",
            "llama":   "meta-llama/Llama-3.1-8B",
            "gemma":   "google/gemma-2-9b"},
}[USE_INSTRUCT]

MODELS = [
    {"name": "mistral", "repo": REPOS["mistral"]},
    {"name": "llama",   "repo": REPOS["llama"]},
    {"name": "gemma",   "repo": REPOS["gemma"], "attn": "eager"},
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

os.makedirs(PROG_DIR, exist_ok=True)
print("scope: {}  ->  {}".format(_scope, OUTPUT_FILE))
print("\nJudges ({}):".format("instruct" if USE_INSTRUCT else "base"))
for m in MODELS:
    cached = DRIVE_OK and os.path.isfile(
        os.path.join(CACHE_DIR, m["name"] + "_4bit", "config.json"))
    print("  {:<8} {:<42} {}".format(
        m["name"], m["repo"], "cached" if cached else "will download"))
print("\ndevice:", "cuda" if torch.cuda.is_available() else "CPU (will be very slow)")

scope: all  ->  comtail_all_difficulty.jsonl

Judges (instruct):
  mistral  mistralai/Mistral-7B-Instruct-v0.3         cached
  llama    meta-llama/Llama-3.1-8B-Instruct           cached
  gemma    google/gemma-2-9b-it                       cached

device: cuda


### Cell 4 - Load, measure per-pair floors, and sample

Two model-free reference points are computed **for every language pair
separately**, using all of that pair's rows in the file rather than only the
sampled ones - the sample gives about 10 rows per pair, far too few for a
stable floor.

- **Copy the source** - what a system scores by emitting the input unchanged.
  Near zero wherever the scripts differ; substantial for `hin-doi` and
  `hin-mar`.
- **An unrelated translation in the same pair** - the true wrong-answer level.

`PAIR_FLOOR[pair]` is the higher of the two, and is what Cell 9 and Cell 11
compare against.

No single pair holds 200 rows - the largest has about 133 - so a narrow
`PAIR_FILTER` scores every row in scope rather than failing. The cell prints the
count it actually used. The printed table is worth reading before the run: a pair
whose copy floor is high is one where a good-looking score may mean nothing.

In [6]:
import sacrebleu
_CHRF = sacrebleu.CHRF(word_order=2)          # word_order=2 makes this chrF++


def chrf_pp(hypothesis, reference):
    if not hypothesis or not str(hypothesis).strip():
        return 0.0
    return _CHRF.sentence_score(str(hypothesis), [str(reference)]).score / 100.0


def usable(row):
    q = str(row.get("question") or "").strip()
    a = str(row.get("answer") or "").strip()
    return bool(q) and len(a.split()) >= MIN_REF_WORDS and q != a


with open(INPUT_FILE, encoding="utf-8") as f:
    all_rows = [json.loads(line) for line in f]
print("Loaded {} rows from {}".format(len(all_rows), INPUT_FILE))

pool = [r for r in all_rows if usable(r)]
if PAIR_FILTER:
    if "-" in PAIR_FILTER:
        pool = [r for r in pool if r["language"] == PAIR_FILTER]
    else:
        pool = [r for r in pool if r["language"].startswith(PAIR_FILTER + "-")]
print("in scope: {} rows across {} pairs".format(
    len(pool), len({r["language"] for r in pool})))
# no single pair has 200 rows (the largest has ~133), so a narrow PAIR_FILTER
# scores everything available rather than failing
N_SAMPLE = min(N_ROWS, len(pool))
if N_SAMPLE < N_ROWS:
    print("  scope holds {} rows - scoring all of them instead of {}".format(
        N_SAMPLE, N_ROWS))
assert N_SAMPLE >= 20, "scope too small to be meaningful ({} rows)".format(N_SAMPLE)

# ---- per-pair floors, from every row of that pair ----
by_pair = defaultdict(list)
for r in pool:
    by_pair[r["language"]].append(r)

rng = random.Random(SEED)
PAIR_COPY, PAIR_RAND, PAIR_FLOOR = {}, {}, {}
for pair, rs in by_pair.items():
    sub = rs if len(rs) <= 60 else rng.sample(rs, 60)
    PAIR_COPY[pair] = statistics.mean(
        chrf_pp(r["question"], r["answer"]) for r in sub)
    PAIR_RAND[pair] = statistics.mean(
        chrf_pp(sub[(i + 3) % len(sub)]["answer"], r["answer"])
        for i, r in enumerate(sub))
    PAIR_FLOOR[pair] = max(PAIR_COPY[pair], PAIR_RAND[pair])

print("\nper-pair floors (chrF++, no model involved):")
print("  {:<10} {:>5} {:>10} {:>11} {:>9}".format(
    "pair", "n", "copy-src", "unrelated", "floor"))
for pair in sorted(by_pair, key=lambda p: -PAIR_COPY[p]):
    warn = "  <- copying scores well here" if pair in HIGH_COPY_PAIRS else ""
    print("  {:<10} {:>5} {:>9.1%} {:>11.1%} {:>9.1%}{}".format(
        pair, len(by_pair[pair]), PAIR_COPY[pair], PAIR_RAND[pair],
        PAIR_FLOOR[pair], warn))

random.seed(SEED)
sample = random.sample(pool, N_SAMPLE)
MAX_FLOOR = max(PAIR_FLOOR[r["language"]] for r in sample)

print("\nSampled {} rows | pairs represented: {}".format(
    len(sample), len({r["language"] for r in sample})))
print("  rows per pair: {}".format(
    dict(Counter(r["language"] for r in sample).most_common(5))), "...")
print("  highest floor among sampled pairs: {:.1%}".format(MAX_FLOOR))

r = sample[0]
print("\n--- example row ---")
print("  pair: {}".format(r["language"]))
print("  src : {}".format(" ".join(str(r["question"]).split())[:92]))
print("  tgt : {}".format(" ".join(str(r["answer"]).split())[:92]))

Loaded 1961 rows from comtail.jsonl
in scope: 1940 rows across 21 pairs

per-pair floors (chrF++, no model involved):
  pair           n   copy-src   unrelated     floor
  hin-doi       80     30.6%       10.6%     30.6%  <- copying scores well here
  hin-mar      125     18.8%       10.8%     18.8%  <- copying scores well here
  eng-tel      125      4.3%       11.5%     11.5%
  eng-tam       91      3.7%       12.5%     12.5%
  eng-hin      109      3.3%       10.4%     10.4%
  eng-kan      103      2.6%       11.5%     11.5%
  hin-tel       66      1.9%       12.5%     12.5%
  eng-guj       87      1.9%       11.9%     11.9%
  hin-kan       92      1.7%       11.4%     11.4%
  hin-guj       87      1.5%       12.2%     12.2%
  hin-pan      102      1.5%        9.9%      9.9%
  eng-mar      117      1.5%       12.3%     12.3%
  eng-pan       71      1.1%       10.7%     10.7%
  hin-odi      103      1.0%       12.1%     12.1%
  hin-ban       60      1.0%       10.0%     10.0%
  eng-b

### Cell 5 - Build the translation prompt

The prompt names both languages explicitly, resolved from the row's own
`language` code - "Translate English into Telugu", not a generic instruction.
With 21 pairs in one file that is essential; a model told only "translate"
will often produce the wrong target language.

**Few-shot examples are drawn from the same language pair**, and from outside
the 200-row sample. A Marathi example teaches nothing about Telugu, and the
target script has to be demonstrated. If a pair has too few spare rows, the
prompt falls back to zero-shot for that pair rather than borrowing examples
from a different language - which would actively mislead.

In [7]:
def flat(text):
    return " ".join(str(text).split())


def pair_of(row):
    src, tgt = row["language"].split("-")
    return LANG_NAMES.get(src, src), LANG_NAMES.get(tgt, tgt)


def pick_fewshot(pair, k=3):
    used = {r["id"] for r in sample}
    cand = [r for r in by_pair.get(pair, [])
            if r["id"] not in used and 5 <= len(str(r["question"]).split()) <= 30]
    random.Random(SEED + 1).shuffle(cand)
    return cand[:k]


FEWSHOT_BY_PAIR = {p: pick_fewshot(p) for p in sorted({r["language"] for r in sample})}
_thin = [p for p, v in FEWSHOT_BY_PAIR.items() if len(v) < 3]
print("few-shot examples per pair (all from OUTSIDE the sample):")
print("  {}".format({p: len(v) for p, v in sorted(FEWSHOT_BY_PAIR.items())}))
if _thin:
    print("  pairs with fewer than 3 spare examples: {}".format(_thin))
    print("  (those run with what is available - never with another pair's examples)")


def shots_for(row):
    return FEWSHOT_BY_PAIR.get(row["language"], [])


def instructions(row):
    s, t = pair_of(row)
    return ("Translate {} into {}.\n\n"
            "Write natural, fluent {} in its own script. Keep proper names and "
            "technical terms in their usual {} forms.\n\n"
            "Output only the {} translation - no transliteration, no "
            "explanation, no repetition of the {}.").format(s, t, t, t, t, s)


def build_completion(row):
    s, t = pair_of(row)
    text = instructions(row) + "\n"
    for ex in shots_for(row):
        text += "\n{}: {}\n{}: {}\n".format(s, flat(ex["question"]),
                                               t, flat(ex["answer"]))
    text += "\n{}: {}\n{}:".format(s, flat(row["question"]), t)
    return text


def build_chat_messages(row):
    s, t = pair_of(row)
    msgs = [{"role": "system", "content": instructions(row)}]
    for ex in shots_for(row):
        msgs.append({"role": "user", "content": flat(ex["question"])})
        msgs.append({"role": "assistant", "content": flat(ex["answer"])})
    msgs.append({"role": "user", "content": flat(row["question"])})
    return msgs


print("\n" + "=" * 70)
print(build_completion(sample[0]))
print("=" * 70)
print("[reference: {}]".format(flat(sample[0]["answer"])))

few-shot examples per pair (all from OUTSIDE the sample):
  {'eng-ban': 3, 'eng-guj': 3, 'eng-hin': 3, 'eng-kan': 3, 'eng-kas': 3, 'eng-mar': 3, 'eng-odi': 3, 'eng-pan': 3, 'eng-tam': 3, 'eng-tel': 3, 'eng-urd': 3, 'hin-ban': 3, 'hin-doi': 3, 'hin-guj': 3, 'hin-kan': 3, 'hin-mar': 3, 'hin-odi': 3, 'hin-pan': 3, 'hin-snd': 3, 'hin-tel': 3, 'hin-urd': 3}

Translate Hindi into Bengali.

Write natural, fluent Bengali in its own script. Keep proper names and technical terms in their usual Bengali forms.

Output only the Bengali translation - no transliteration, no explanation, no repetition of the Hindi.

Hindi: केवल मूर्ख ही सोच सकते हैं कि सरकार उस व्यक्ति के साथ मैत्रीपूर्ण रहेगी जो घर-घर में अल्लामा सईदी बनाने के लिए खुले तौर पर प्रार्थना करता है!
Bengali: যে ব্যক্তি ঘরে ঘরে আল্লামা সাঈদী বানানোর জন্য প্রকাশ্যে দোয়া করেন, তাঁর সাথে সরকারের প্রীতি বজায় থাকবে এটা বোকারাই ভাবতে পারে!

Hindi: बीएनपी अब उनकी रिहाई की मांग करने वाले शांतिपूर्ण कार्यक्रमों के साथ मैदान में उतरने में असमर्थ ह

### Cell 6 - Generate and score

Greedy decoding for reproducibility, with `max_new_tokens` scaled to the source
length.

`clean_translation` strips the label a model prepends (`Telugu:`,
`Translation:`) and cuts anything from a following source-language marker,
which base models emit as they continue the few-shot pattern. The label is
built from the row's own target language, so it works across all 21 pairs.

It deliberately does **not** repair wrong-script output. A model that answers in
the source language has failed the task and should score near zero; Cell 11
reports how often that happens, since a high rate is a prompting problem rather
than a difficulty signal.

In [8]:
_GENERIC_LABEL = re.compile(
    r"^\s*(here(?:\s+is|'s)?\s+the\s+)?(translation|output)\s*[:\-]\s*", re.I)
_LANG_LABEL = re.compile(
    r"^\s*(" + "|".join(sorted(set(LANG_NAMES.values()))) + r")\s*[:\-]\s*", re.I)
_SRC_MARKER = re.compile(
    r"\n\s*(" + "|".join(sorted(set(LANG_NAMES.values()))) + r")\s*:", re.I)


def clean_translation(text):
    t = (text or "").strip()
    t = _SRC_MARKER.split(t)[0]              # a base model continuing the pattern
    for line in t.split("\n"):
        line = line.strip()
        if not line:
            continue
        line = _GENERIC_LABEL.sub("", line)
        line = _LANG_LABEL.sub("", line)
        line = line.strip().strip('"\u2018\u2019\u201c\u201d')
        if line:
            return " ".join(line.split())
    return ""


def prompt_style(tokenizer):
    # base checkpoints have no chat template at all
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


def looks_like_source(hyp, row):
    """High overlap with the source suggests it echoed rather than translated."""
    return chrf_pp(hyp, row["question"]) > 0.80


@torch.no_grad()
def translate(model, tokenizer, row):
    if prompt_style(tokenizer) == "completion":
        text = build_completion(row)
    else:
        msgs = build_chat_messages(row)
        try:
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            # some templates (Gemma) reject a system role - fold it into the
            # first user turn rather than dropping the instructions
            merged = [dict(m) for m in msgs[1:]]
            merged[0]["content"] = msgs[0]["content"] + "\n\n" + merged[0]["content"]
            text = tokenizer.apply_chat_template(
                merged, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    n_in   = inputs["input_ids"].shape[1]
    budget = min(4 * len(str(row["question"]).split()) + 48, MAX_NEW_TOKENS)
    out = model.generate(**inputs,
                         max_new_tokens=budget,
                         do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return clean_translation(tokenizer.decode(out[0][n_in:], skip_special_tokens=True))


def clear_hf_cache():
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()


for raw, want in [
    ("Telugu: hello", "hello"),
    ("Translation: hi there", "hi there"),
    ("good\nEnglish: next", "good"),
    ('"quoted"', "quoted"),
    ("", ""),
]:
    got = clean_translation(raw)
    assert got == want, (raw, got, want)
print("Generation and scoring functions defined")

Generation and scoring functions defined


### Cell 7 - Load-or-cache, and the batched runner

`load_model` implements download-once against the Drive cache; a checkpoint
restored from Drive is already 4-bit, so a fresh `BitsAndBytesConfig` would
conflict and is omitted. `trust_remote_code` stays off.

`run_model` stores the raw chrF++ score, the output, and whether the output
merely echoed the source - never a pass/fail, which is what makes Cell 9
re-runnable for free. Each batch is appended before the next begins, so a
disconnect costs at most `BATCH_SIZE` rows.

In [9]:
def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached     = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source     = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs skip the download")

    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9, prompt_style(tokenizer)))
    return model, tokenizer


def run_model(spec, rows):
    prog = os.path.join(PROG_DIR, spec["name"] + ".jsonl")

    done = {}
    if os.path.exists(prog):
        with open(prog, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item
        print("  resuming - {}/{} already scored".format(len(done), len(rows)))

    remaining = [r for r in rows if r["id"] not in done]
    if not remaining:
        print("  {} already complete - skipping load".format(spec["name"]))
        return done

    model, tokenizer = load_model(spec)
    total_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for start in range(0, len(remaining), BATCH_SIZE):
        batch, results = remaining[start:start + BATCH_SIZE], []
        for row in batch:
            hyp = translate(model, tokenizer, row)
            results.append({
                "id":      row["id"],
                "pair":    row["language"],
                "chrf":    chrf_pp(hyp, row["answer"]),
                "n_words": len(hyp.split()),
                "echoed":  int(looks_like_source(hyp, row)),
                "output":  hyp,
            })
        with open(prog, "a", encoding="utf-8") as f:
            for item in results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        done.update({i["id"]: i for i in results})
        mean = sum(v["chrf"] for v in done.values()) / len(done)
        print("  batch {}/{} saved - {}/{} rows | mean chrF++ {:.1%}".format(
            start // BATCH_SIZE + 1, total_batches, len(done), len(rows), mean))

    del model, tokenizer
    clear_hf_cache()
    print("  {} complete".format(spec["name"]))
    return done

print("Runner defined")

Runner defined


### Cell 8 - Run all three models

One at a time - loaded, scored, unloaded - so peak VRAM stays near 6 GB. Budget
roughly 8-12 min per model plus downloads on the first run. Safe to re-run;
anything already scored is skipped.

In [10]:
preds = {}
for spec in MODELS:
    print("\n=== {} ===".format(spec["name"]))
    preds[spec["name"]] = run_model(spec, sample)

print("\nAll models done")


=== mistral ===
  loading /drive/MyDrive/models/mistral_4bit [Drive cache (already 4-bit)]


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ready | VRAM: 4.14GB | prompt style: chat
  batch 1/8 saved - 25/200 rows | mean chrF++ 15.6%
  batch 2/8 saved - 50/200 rows | mean chrF++ 16.1%
  batch 3/8 saved - 75/200 rows | mean chrF++ 18.2%
  batch 4/8 saved - 100/200 rows | mean chrF++ 17.8%
  batch 5/8 saved - 125/200 rows | mean chrF++ 17.8%
  batch 6/8 saved - 150/200 rows | mean chrF++ 17.6%
  batch 7/8 saved - 175/200 rows | mean chrF++ 17.1%
  batch 8/8 saved - 200/200 rows | mean chrF++ 17.0%
  mistral complete

=== llama ===
  loading /drive/MyDrive/models/llama_4bit [Drive cache (already 4-bit)]


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ready | VRAM: 5.71GB | prompt style: chat


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  batch 1/8 saved - 25/200 rows | mean chrF++ 32.5%
  batch 2/8 saved - 50/200 rows | mean chrF++ 33.2%
  batch 3/8 saved - 75/200 rows | mean chrF++ 36.0%
  batch 4/8 saved - 100/200 rows | mean chrF++ 34.9%
  batch 5/8 saved - 125/200 rows | mean chrF++ 34.7%
  batch 6/8 saved - 150/200 rows | mean chrF++ 33.6%
  batch 7/8 saved - 175/200 rows | mean chrF++ 33.6%
  batch 8/8 saved - 200/200 rows | mean chrF++ 33.6%
  llama complete

=== gemma ===
  loading /drive/MyDrive/models/gemma_4bit [Drive cache (already 4-bit), attn=eager]


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

  ready | VRAM: 6.14GB | prompt style: chat
  batch 1/8 saved - 25/200 rows | mean chrF++ 36.3%
  batch 2/8 saved - 50/200 rows | mean chrF++ 36.8%
  batch 3/8 saved - 75/200 rows | mean chrF++ 38.7%
  batch 4/8 saved - 100/200 rows | mean chrF++ 37.7%
  batch 5/8 saved - 125/200 rows | mean chrF++ 37.7%
  batch 6/8 saved - 150/200 rows | mean chrF++ 37.0%
  batch 7/8 saved - 175/200 rows | mean chrF++ 37.3%
  batch 8/8 saved - 200/200 rows | mean chrF++ 37.5%
  gemma complete

All models done


### Cell 9 - Find the threshold

One threshold is applied across all pairs. That is a deliberate compromise:
with ~10 sampled rows per pair there is not enough data to derive 21 separate
thresholds, so the run uses a global one and Cell 11 exposes the per-pair
consequences rather than hiding them.

`"auto_median"` takes the pooled median. `"floor_margin"` anchors to the
**highest floor among the sampled pairs** plus `FLOOR_MARGIN` - use it whenever
`hin-doi` or `hin-mar` are in scope, since a threshold below their copy floor
would pass a model that merely echoed the Hindi.

Warnings fire if the chosen threshold falls below that highest floor, sits so
high only near-exact matches pass, or empties a difficulty band.

In [11]:
pooled = sorted(preds[s["name"]][r["id"]]["chrf"] for r in sample for s in MODELS)

def pct(p):
    return pooled[min(len(pooled) - 1, int(p * len(pooled)))]

print("chrF++ distribution per model:")
print("  {:<10} {:>7} {:>7} {:>7} {:>7}".format("model", "p25", "median", "p75", "mean"))
for s in MODELS:
    v = sorted(x["chrf"] for x in preds[s["name"]].values())
    print("  {:<10} {:>6.1%} {:>7.1%} {:>7.1%} {:>7.1%}".format(
        s["name"], v[len(v) // 4], v[len(v) // 2], v[3 * len(v) // 4],
        sum(v) / len(v)))
print("  {:<10} {:>6.1%} {:>7.1%} {:>7.1%} {:>7.1%}".format(
    "POOLED", pct(.25), pct(.50), pct(.75), sum(pooled) / len(pooled)))

print("\nhighest per-pair floor among sampled pairs: {:.1%}".format(MAX_FLOOR))


def difficulty_at(th):
    out = Counter()
    for r in sample:
        votes = sum(preds[s["name"]][r["id"]]["chrf"] >= th for s in MODELS)
        out["Easy" if votes == 3 else ("Medium" if votes == 2 else "Hard")] += 1
    return out


if THRESHOLD_MODE == "auto_median":
    THRESHOLD = pct(.50)
    why = "median of all pooled model scores"
elif THRESHOLD_MODE == "floor_margin":
    THRESHOLD = MAX_FLOOR + FLOOR_MARGIN
    why = "highest per-pair floor + {:.2f}".format(FLOOR_MARGIN)
elif THRESHOLD_MODE == "fixed":
    THRESHOLD = FIXED_THRESHOLD
    why = "FIXED_THRESHOLD from Cell 3"
else:
    raise ValueError("unknown THRESHOLD_MODE: " + str(THRESHOLD_MODE))

print("\nsensitivity - what each threshold would produce:")
print("  {:>9}  {:>6} {:>7} {:>6}".format("threshold", "Easy", "Medium", "Hard"))
for th in sorted(set(round(x, 3) for x in
                     [.20, .30, .40, .50, .60, .70,
                      round(MAX_FLOOR, 3), round(THRESHOLD, 3)])):
    d = difficulty_at(th)
    tag = ""
    if abs(th - round(THRESHOLD, 3)) < 1e-9:
        tag += "  <- CHOSEN"
    if abs(th - round(MAX_FLOOR, 3)) < 1e-9:
        tag += "  (highest pair floor)"
    print("  {:>9.3f}  {:>6} {:>7} {:>6}{}".format(
        th, d.get("Easy", 0), d.get("Medium", 0), d.get("Hard", 0), tag))

print("\nTHRESHOLD = {:.3f}  ({})".format(THRESHOLD, why))
if THRESHOLD <= MAX_FLOOR:
    print("  WARNING: below the highest per-pair floor ({:.1%}). On that pair a".format(MAX_FLOOR))
    print("  model could pass without translating. Use THRESHOLD_MODE='floor_margin'.")
else:
    print("  sits {:.1f} points above the highest per-pair floor - OK".format(
        100 * (THRESHOLD - MAX_FLOOR)))
if THRESHOLD >= 0.95:
    print("  WARNING: near the top of the range - only near-exact matches pass.")
_b = difficulty_at(THRESHOLD)
if min(_b.get(k, 0) for k in ("Easy", "Medium", "Hard")) == 0:
    print("  WARNING: one difficulty band is empty - pick another value.")

chrF++ distribution per model:
  model          p25  median     p75    mean
  mistral      8.5%   15.6%   24.0%   17.0%
  llama       21.6%   32.6%   44.8%   33.6%
  gemma       24.7%   36.4%   50.1%   37.5%
  POOLED      15.5%   27.3%   41.1%   29.3%

highest per-pair floor among sampled pairs: 30.6%

sensitivity - what each threshold would produce:
  threshold    Easy  Medium   Hard
      0.200      67      80     53
      0.273      27      87     86  <- CHOSEN
      0.300      23      75    102
      0.306      20      76    104  (highest pair floor)
      0.400       5      47    148
      0.500       1      19    180
      0.600       0       8    192
      0.700       0       1    199

THRESHOLD = 0.273  (median of all pooled model scores)
  model could pass without translating. Use THRESHOLD_MODE='floor_margin'.


### Cell 10 - Apply the threshold and write the schema

Votes sum into Easy / Medium / Hard as in every other split. Output rows are
rebuilt key-by-key from `SCHEMA_KEYS`, so all 14 fields survive in schema order
and `explanation` stays `null` - the unreliable alignment score is not
reintroduced. The audit file records each row's language pair and that pair's
floor, so a suspicious label can be traced.

In [12]:
def get_difficulty(votes):
    score = sum(votes)
    if score == 3:
        return "Easy"
    elif score == 2:
        return "Medium"
    else:
        return "Hard"


final_results, audit = [], []

for row in sample:
    scores = [preds[s["name"]][row["id"]]["chrf"] for s in MODELS]
    votes  = [int(x >= THRESHOLD) for x in scores]
    difficulty = get_difficulty(votes)

    enriched = {**row, "difficulty": difficulty}
    if SET_EVAL_METRIC:
        enriched["eval_metric"] = SET_EVAL_METRIC
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id":           row["id"],
        "pair":         row["language"],
        "pair_floor":   round(PAIR_FLOOR[row["language"]], 4),
        "difficulty":   difficulty,
        "votes":        votes,
        "chrf":         [round(x, 4) for x in scores],
        "threshold":    round(THRESHOLD, 4),
        "source":       " ".join(str(row["question"]).split()),
        "reference":    " ".join(str(row["answer"]).split()),
        "translations": {s["name"]: preds[s["name"]][row["id"]]["output"]
                         for s in MODELS},
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}".format(AUDIT_FILE))
print("threshold used: {:.3f}".format(THRESHOLD))

Saved -> comtail_all_difficulty.jsonl (200 rows)
Audit -> comtail_all_audit.jsonl
threshold used: 0.273


### Cell 11 - Verify and report

1. **Schema** - 14 keys in order, no null `difficulty`, `explanation` still
   null, and source/target text identical to the input.
2. **Difficulty distribution.**
3. **Per-model mean chrF++**, and how often the output merely echoed the
   source - the failure the two Devanagari pairs invite.
4. **Per-pair accuracy against that pair's own floor.** This is the table that
   matters for this file. A pair whose mean score barely clears its floor is
   not being translated, however healthy the global average looks - and
   `hin-doi` and `hin-mar` can flatter a model that only copies.

With ~10 rows per pair these per-pair numbers are indicative, not conclusive.
Re-run with `PAIR_FILTER` set to a single pair for a real read on it.

In [13]:
bad_keys = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
missing  = [r["id"] for r in final_results if r["difficulty"] is None]
src_by_id = {r["id"]: r for r in all_rows}
altered = [r["id"] for r in final_results
           if r["question"] != src_by_id[r["id"]]["question"]
           or r["answer"] != src_by_id[r["id"]]["answer"]]
print("Schema check : {} rows | wrong keys: {} | null difficulty: {}".format(
    len(final_results), len(bad_keys), len(missing)))
print("source/target altered: {} | explanation non-null: {}".format(
    len(altered), sum(1 for r in final_results if r["explanation"] is not None)))

dist  = Counter(r["difficulty"] for r in final_results)
total = len(final_results)
print("\nDifficulty distribution (threshold {:.3f}):".format(THRESHOLD))
for level in ["Easy", "Medium", "Hard"]:
    n = dist.get(level, 0)
    print("  {:<7}: {:4d}  ({:.1f}%)".format(level, n, n / total * 100))

print("\nPer-model:")
for s in MODELS:
    v = list(preds[s["name"]].values())
    m = sum(x["chrf"] for x in v) / len(v)
    echo = sum(x["echoed"] for x in v)
    empty = sum(1 for x in v if x["n_words"] == 0)
    flags = []
    if echo / total > 0.10:
        flags.append("echoing the source")
    if empty / total > 0.05:
        flags.append("empty outputs")
    print("  {:<10} mean chrF++ {:.1%} | echoed {:>3}/{} | empty {:>3}/{}{}".format(
        s["name"], m, echo, total, empty, total,
        "  <- " + ", ".join(flags) if flags else ""))

print("\nPer-pair (mean chrF++ vs that pair's own floor):")
print("  {:<10} {:>4} {:>8}".format("pair", "n", "floor") +
      "".join("{:>10}".format(s["name"][:8]) for s in MODELS))
pairs = Counter(r["language"] for r in sample)
for pair, n in sorted(pairs.items()):
    ids = [r["id"] for r in sample if r["language"] == pair]
    line = "  {:<10} {:>4} {:>7.1%}".format(pair, n, PAIR_FLOOR[pair])
    weak = False
    for s in MODELS:
        m = sum(preds[s["name"]][i]["chrf"] for i in ids) / n
        line += "{:>10.1%}".format(m)
        if m <= PAIR_FLOOR[pair]:
            weak = True
    if weak:
        line += "  <- at/below floor"
    print(line)

print("\n--- 2 sample rows ---")
for a in audit[:2]:
    print("\n  {} [{}] {} chrf={}".format(a["id"], a["pair"], a["difficulty"], a["chrf"]))
    print("    src: {}".format(a["source"][:88]))
    print("    ref: {}".format(a["reference"][:88]))
    for k, v in a["translations"].items():
        print("    {:<8}: {}".format(k, (v or "<empty>")[:88]))

Schema check : 200 rows | wrong keys: 0 | null difficulty: 0
source/target altered: 0 | explanation non-null: 0

Difficulty distribution (threshold 0.273):
  Easy   :   27  (13.5%)
  Medium :   87  (43.5%)
  Hard   :   86  (43.0%)

Per-model:
  mistral    mean chrF++ 17.0% | echoed   1/200 | empty   0/200
  llama      mean chrF++ 33.6% | echoed   4/200 | empty   0/200
  gemma      mean chrF++ 37.5% | echoed   1/200 | empty   0/200

Per-pair (mean chrF++ vs that pair's own floor):
  pair          n    floor   mistral     llama     gemma
  eng-ban      11   10.9%     20.2%     34.6%     40.3%
  eng-guj      10   11.9%     14.8%     42.6%     43.9%
  eng-hin      17   10.4%     29.3%     48.5%     48.6%
  eng-kan       8   11.5%     14.6%     27.0%     39.9%
  eng-kas      11   11.1%      9.7%     10.2%      9.8%  <- at/below floor
  eng-mar      12   12.3%     17.8%     35.5%     40.5%
  eng-odi       2   11.1%      4.3%     14.6%     24.7%  <- at/below floor
  eng-pan       7   10.7%   

In [14]:
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    print("not downloaded")

not download
